<a href="https://colab.research.google.com/github/nakyung55/FintechAI/blob/main/5_%EA%B0%9C%EC%9D%B8%EB%8C%80%EC%B6%9C%EC%8A%B9%EC%9D%B8%EC%98%88%EC%B8%A1%EB%AA%A8%EB%8D%B8%EA%B5%AC%EC%B6%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **개인대출승인예측모델구축**


- 모델의 성능 지표 : ROC AUC
- 데이터는 5,000건이며, 총 14개의 컬럼으로 구성됨
- Personal Loan(개인대출전환여부) 컬럼이 예측값임
- ID : 고객ID
- Age : 고객나이
- Experience : 직장 경험
- Income : 수입액
- ZIP Code : 우편번호
- Family : 가족수
- CCAvg : 월평균카드 사용액
- Education : 학력
- Mortgage : 주택담보 대출 금액
- Personal Loan : 개인대출 전환여부
- Securities Account : 보험 유무
- CD Account : 양도성 예금 증서 보유 여부
- Online : 온라인 뱅킹 사용 여부
- CreditCard : 신용카드 보유 여부

In [28]:
import gdown
import pandas as pd
import numpy as np
from tqdm import tqdm
import gc
import requests
import pickle
from io import StringIO
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
from sklearn.model_selection import train_test_split

In [19]:
web_url = "https://raw.githubusercontent.com/trex99/fintech/main/train.csv"
response = requests.get(web_url)
data = StringIO(response.text)
train = pd.read_csv(data)
print("데이터 shape:", train.shape)
print(train.head())

# ID, ZIP Code 컬럼 제거
drop_cols = []
if 'ID' in train.columns:
    drop_cols.append('ID')
if 'ZIP Code' in train.columns:
    drop_cols.append('ZIP Code')
train = train.drop(columns=drop_cols)
print("컬럼 제거 후 shape:", train.shape)

데이터 shape: (5000, 14)
   ID  Age  Experience  Income  ZIP Code  Family  CCAvg  Education  Mortgage  \
0   1   25           1      49     91107       4    1.6          1         0   
1   2   45          19      34     90089       3    1.5          1         0   
2   3   39          15      11     94720       1    1.0          1         0   
3   4   35           9     100     94112       1    2.7          2         0   
4   5   35           8      45     91330       4    1.0          2         0   

   Personal Loan  Securities Account  CD Account  Online  CreditCard  
0              0                   1           0       0           0  
1              0                   1           0       0           0  
2              0                   0           0       0           0  
3              0                   0           0       0           0  
4              0                   0           0       0           1  
컬럼 제거 후 shape: (5000, 12)


In [20]:
train.info()
train.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 5000 non-null   int64  
 1   Experience          5000 non-null   int64  
 2   Income              5000 non-null   int64  
 3   Family              5000 non-null   int64  
 4   CCAvg               5000 non-null   float64
 5   Education           5000 non-null   int64  
 6   Mortgage            5000 non-null   int64  
 7   Personal Loan       5000 non-null   int64  
 8   Securities Account  5000 non-null   int64  
 9   CD Account          5000 non-null   int64  
 10  Online              5000 non-null   int64  
 11  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(11)
memory usage: 468.9 KB


,count,mean,std,min,25%,50%,75%,max
Age,5000.0,45.338400,11.463166,23.0,35.0,45.0,55.0,67.0
Experience,5000.0,20.104600,11.467954,-3.0,10.0,20.0,30.0,43.0
Income,5000.0,73.774200,46.033729,8.0,39.0,64.0,98.0,224.0
Family,5000.0,2.396400,1.147663,1.0,1.0,2.0,3.0,4.0
CCAvg,5000.0,1.937938,1.747659,0.0,0.7,1.5,2.5,10.0
Education,5000.0,1.881000,0.839869,1.0,1.0,2.0,3.0,3.0
Mortgage,5000.0,56.498800,101.713802,0.0,0.0,0.0,101.0,635.0
Personal Loan,5000.0,0.096000,0.294621,0.0,0.0,0.0,0.0,1.0
Securities Account,5000.0,0.104400,0.305809,0.0,0.0,0.0,0.0,1.0
CD Account,5000.0,0.060400,0.238250,0.0,0.0,0.0,0.0,1.0


In [21]:
train['use_income_by_family'] = train['Income'] / train['Family'] # 가족 대비 소득
if 'CCAvg' in train.columns:
    train['ccavg_per_income'] = train['CCAvg'] / train['Income'] # 소비 대비 소득

# Feature / Target 분리
X = train.drop(columns=['Personal Loan'])
y = train['Personal Loan']

In [22]:
x_train, x_valid, y_train, y_valid = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)
x_train.shape, x_valid.shape, y_train.shape, y_valid.shape

# 예측 후 결과 비교를 위해서 복제
x_valid_include_pred = x_valid.copy()

In [23]:
# 학습
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
print("scale_pos_weight:", scale_pos_weight)

params = {
    'n_estimators': 3000,
    'learning_rate': 0.01,
    'max_depth': 4,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight
}

model = XGBClassifier(
    **params,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)],
    verbose=False
)

scale_pos_weight: 9.416666666666666


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=3000,
              n_jobs=None, num_parallel_tree=None, ...)

In [24]:
# 예측
y_pred = model.predict_proba(x_valid)[:, 1]
x_valid_include_pred['pred'] = y_pred
x_valid_include_pred

,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Securities Account,CD Account,Online,CreditCard,use_income_by_family,ccavg_per_income,pred
2388,64,39,23,3,0.5,1,0,1,0,0,0,7.666667,0.021739,5.140957e-07
2373,33,9,184,2,4.8,2,0,0,0,0,0,92.000000,0.026087,9.996054e-01
4347,58,33,22,3,0.2,1,0,0,0,1,0,7.333333,0.009091,3.186368e-06
665,54,24,61,4,2.0,3,0,1,0,1,0,15.250000,0.032787,7.169470e-06
4182,55,29,49,2,0.8,3,220,0,0,0,1,24.500000,0.016327,1.769374e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1590,49,23,58,4,2.6,1,188,0,0,0,0,14.500000,0.044828,1.027956e-04
4986,32,6,78,1,2.9,3,0,0,0,0,0,78.000000,0.037179,2.268317e-03
433,52,28,31,4,0.2,1,141,0,0,1,1,7.750000,0.006452,3.141156e-05
1821,32,7,54,4,1.3,1,0,1,0,1,0,13.500000,0.024074,2.203135e-06


In [25]:
# 검증
score = roc_auc_score(y_valid, y_pred)
print('ROC AUC Score = ', score)

best_f1 = 0
best_thresh = 0
for t in np.arange(0.1, 0.9, 0.01):
    preds = (y_pred > t).astype(int)
    f1 = f1_score(y_valid, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
print("Best F1:", best_f1)
print("Best Threshold:", best_thresh)

ROC AUC Score =  0.9994008112094396
Best F1: 0.9494949494949495
Best Threshold: 0.4299999999999998


In [26]:
# 변수 중요도
val_imp = pd.DataFrame(
    model.feature_importances_,
    index=x_train.columns,
    columns=['imp']
)

val_imp = val_imp.sort_values(by='imp', ascending=False)
val_imp

,imp
Income,0.312847
Family,0.187464
Education,0.151579
CCAvg,0.093004
CD Account,0.063602
use_income_by_family,0.057559
ccavg_per_income,0.038313
Securities Account,0.020006
CreditCard,0.018394
Age,0.015521


In [29]:
save_object = [model, params, x_valid_include_pred]
with open(file='my_model.pickle', mode='wb') as f:
    pickle.dump(save_object, f)
with open(file='my_model.pickle', mode='rb') as f:
    load_object = pickle.load(f)

model = load_object[0]
params = load_object[1]
valid_data = load_object[2]

valid_data['pred'] = model.predict_proba(x_valid)[:, 1]
valid_data

,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Securities Account,CD Account,Online,CreditCard,use_income_by_family,ccavg_per_income,pred
2388,64,39,23,3,0.5,1,0,1,0,0,0,7.666667,0.021739,5.140957e-07
2373,33,9,184,2,4.8,2,0,0,0,0,0,92.000000,0.026087,9.996054e-01
4347,58,33,22,3,0.2,1,0,0,0,1,0,7.333333,0.009091,3.186368e-06
665,54,24,61,4,2.0,3,0,1,0,1,0,15.250000,0.032787,7.169470e-06
4182,55,29,49,2,0.8,3,220,0,0,0,1,24.500000,0.016327,1.769374e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1590,49,23,58,4,2.6,1,188,0,0,0,0,14.500000,0.044828,1.027956e-04
4986,32,6,78,1,2.9,3,0,0,0,0,0,78.000000,0.037179,2.268317e-03
433,52,28,31,4,0.2,1,141,0,0,1,1,7.750000,0.006452,3.141156e-05
1821,32,7,54,4,1.3,1,0,1,0,1,0,13.500000,0.024074,2.203135e-06


In [30]:
score = roc_auc_score(y_valid,y_pred)
print('ROC AUC Score = ', score)

ROC AUC Score =  0.9994008112094396


# 개발 방향
- 개발자 도구를 통해 확인한 textarea 태그는 브라우저에서 JavaScript 실행 후 동적으로 생성되지만, requests와 BeautifulSoup는 정적 HTML만 수집 가능하며 JavaScript 실행 결과는 포함하지 않는다. 따라서 selenium을 사용해야만 접근이 가능하지만, 이는 과도한 방식이라고 판단해 직접 태그를 파싱하는 방식 대신 Raw 데이터 경로를 활용했다.
- x_train과 x_valid에 Personal Loan 컬럼이 포함되어 있어 ROC AUC = 1.0이므로 x_train에서 해당 컬럼을 제외한다.
- ID와 ZIP Code 컬럼 제거, ccavg_per_income 파생변수 생성, scale_pos_weight로 클래스 불균형 보정, learning_rate을 낮추고 max_depth를 높여 성능 개선
<br>

# 결과 해석
ROC AUC Score이 1에 매우 가깝게 나타났으며, 이는 대출 승인 여부를 잘 구분하고 있음을 의미한다. Best F1 약 0.94, Best Threshold 약 0.43으로, 대출 승인 비율이 상대적으로 낮은 불균형 구조에서도 안정적인 분류 성능을 보이고 있음을 알 수 있다.
